# Prepare the raw time series

Run all cells from the repository root or `bases/`. Requires Python and pandas.
Reads `bases/raw/` and saves model-ready CSVs in `bases/`, with `Date` first and the original target name retained. Microsoft is also rebuilt from its raw file on every run, matching the established output schema.

| Output | Target | Time unit / processing |
| --- | --- | --- |
| `daily_delhi_climate_prepared.csv` | `meantemp` | Daily; merge train and test, test wins on overlapping dates |
| `pilgrims_pride_prepared.csv` | `Close` | Observed trading sessions; preserve local calendar date |
| `brazil_prepared.csv` | `victories` | Annual; count `Result == W`, including zero-win years |
| `sales_prepared.csv` | `Profit` | Daily; sum all sales rows on each date |
| `microsoft_stock_prepared.csv` | `target_open` | Rebuilt from raw; one-session Close/Volume lags |

No scaling, differencing, feature selection or train/test split is performed here. Keep chronological splits and fit learned transformations within the training window in `pipeline.ipynb`. Climate covariates and same-session stock columns are not known in advance; only use them as lagged exogenous variables. Stock horizons count observed sessions, including exchange-calendar gaps; do not apply `asfreq("B")` to fill holiday prices.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "bases" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook inside the series-temporais repository.")
RAW = ROOT / "bases" / "raw"
OUT = ROOT / "bases"
OUT.mkdir(exist_ok=True)
prepared = {}
summary = []

def read_raw(filename, required):
    frame = pd.read_csv(RAW / filename)
    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"{filename}: missing columns {sorted(missing)}")
    print(f"{filename}: {len(frame):,} source rows")
    return frame

def numeric(frame, columns):
    frame = frame.copy()
    for col in columns:
        frame[col] = pd.to_numeric(frame[col], errors="raise")
    return frame.replace([np.inf, -np.inf], np.nan)

def dated(frame, column, date_format):
    frame = frame.copy()
    frame["Date"] = pd.to_datetime(frame[column], format=date_format, errors="raise").dt.normalize()
    if frame["Date"].isna().any():
        raise ValueError("Missing source dates")
    return frame.drop(columns=[column] if column != "Date" else []).set_index("Date").sort_index()

def save_series(name, frame, target, unit, write=True):
    frame = frame.sort_index().copy()
    frame.index.name = "Date"
    if frame.empty or frame.index.has_duplicates or frame.index.isna().any():
        raise ValueError(f"{name}: empty series or invalid/duplicate dates")
    if not isinstance(frame.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: expected a DatetimeIndex")
    if not pd.api.types.is_numeric_dtype(frame[target]) or not np.isfinite(frame[target]).all():
        raise ValueError(f"{name}: target must be finite and numeric")
    if write:
        frame.to_csv(OUT / name, index=True, date_format="%Y-%m-%d")
    prepared[name] = frame
    summary.append({"file": name, "target": target, "time_unit": unit,
                    "rows": len(frame), "start": frame.index.min().date().isoformat(),
                    "end": frame.index.max().date().isoformat(),
                    "missing_target": int(frame[target].isna().sum()),
                    "action": "saved" if write else "reused"})
    return frame

## Delhi climate — daily mean temperature

The original files overlap on 2017-01-01 and disagree. Concatenate train first, test second, and keep the test observation. Exact duplicate rows are removed; any other within-file date conflict is rejected. Daily gaps or missing numeric measurements use only previous observations (`ffill`), never future observations. Leading missing targets cause an error. Pressure and other covariates are retained as supplied, including outliers; they are not used to determine or alter temperature.


In [ ]:
climate_columns = ["meantemp", "humidity", "wind_speed", "meanpressure"]
parts = []
for filename in ["DailyDelhiClimateTrain.csv", "DailyDelhiClimateTest.csv"]:
    part = read_raw(filename, ["date", *climate_columns]).drop_duplicates()
    part = dated(numeric(part, climate_columns), "date", "%Y-%m-%d")
    if part.index.has_duplicates:
        raise ValueError(f"{filename}: conflicting observations on the same day")
    parts.append(part[climate_columns])
overlap = parts[0].index.intersection(parts[1].index)
print(f"Overlapping dates (test takes precedence): {list(overlap.strftime('%Y-%m-%d'))}")
climate = pd.concat(parts)
climate = climate.loc[~climate.index.duplicated(keep="last")].sort_index()
climate = climate.asfreq("D").ffill()
climate = save_series("daily_delhi_climate_prepared.csv", climate, "meantemp", "D")
climate.head()

## Pilgrim’s Pride — closing price

The source timestamps include Eastern offsets that change with daylight saving time. Extract the source calendar date without shifting it to another timezone. Preserve observed sessions and all numeric market columns. Identical rows can be removed, but conflicting same-date quotes and missing prices cause an error. No synthetic weekend/holiday prices are created.


In [ ]:
stock_file = "Pilgrim_s_Pride_Corporation_1987-12-30_2026-09-16.csv"
stock = read_raw(stock_file, ["Date", "Close"]).drop_duplicates()
stock["Date"] = stock["Date"].str.slice(0, 10)
stock = dated(stock, "Date", "%Y-%m-%d")
stock = numeric(stock, stock.columns)
stock = save_series("pilgrims_pride_prepared.csv", stock, "Close", "observed trading sessions")
stock.head()

## Brazil — victories per calendar year

`W` is counted as a victory according to the dataset’s recorded result, including its classification of shootouts. Count every valid listed match without deduplicating by value. The malformed 2021-09-05 Brazil v Argentina row has `FIFA World Cup` in `Result` and no score; exclude it because it records no usable outcome. Other unknown outcomes cause an error. Reindex the complete year range so years with no listed matches have zero recorded victories. The first/last years and the historical match coverage may be incomplete.


In [ ]:
matches = read_raw("brazil.csv", ["Date", "Result", "Match", "Score"])
matches = dated(matches, "Date", "%d %b %Y")
malformed = ((matches.index == pd.Timestamp("2021-09-05"))
             & matches["Match"].eq("Brazil v Argentina")
             & matches["Result"].eq("FIFA World Cup") & matches["Score"].isna())
print(f"Excluded rows with no usable outcome: {int(malformed.sum())}")
matches = matches.loc[~malformed]
result = matches["Result"].astype("string").str.strip().str.upper()
if not result.isin(["W", "D", "L"]).all():
    raise ValueError("Brazil: missing or unknown match result")
years = range(matches.index.year.min(), matches.index.year.max() + 1)
wins = result.eq("W").groupby(matches.index.year).sum().reindex(years, fill_value=0)
brazil = pd.DataFrame({"victories": wins.to_numpy(dtype=int)},
                      index=pd.to_datetime([f"{year}-01-01" for year in years]))
brazil = brazil.asfreq("YS")
assert int(brazil["victories"].sum()) == int(result.eq("W").sum())
brazil = save_series("brazil_prepared.csv", brazil, "victories", "YS")
brazil.head()

## Sales — total daily profit

Multiple rows on the same date are transactions, so their `Profit` values are summed. Identical-looking transactions are retained because no transaction ID establishes that they are accidental duplicates. Missing or nonfinite profit causes an error. Days with no source rows represent zero recorded profit, assuming the supplied transaction ledger covers the date range. Negative profits remain valid. Transaction attributes are omitted from the univariate daily output.


In [ ]:
sales = read_raw("Sales.csv", ["Date", "Profit"])
sales = dated(numeric(sales, ["Profit"]), "Date", "%Y-%m-%d")
if not np.isfinite(sales["Profit"]).all():
    raise ValueError("Sales: missing or nonfinite transaction profit")
daily_profit = sales[["Profit"]].groupby(level=0).sum().asfreq("D", fill_value=0)
assert np.isclose(daily_profit["Profit"].sum(), sales["Profit"].sum())
sales_prepared = save_series("sales_prepared.csv", daily_profit, "Profit", "D")
sales_prepared.head()

## Microsoft — prepare from raw on every run

Read `Microsoft_Stock.csv`, sort by the source date, and normalize the timestamps to calendar dates. Use the current session’s `Open` as `target_open` and the previous observed trading session’s `Close` and `Volume` as `close_lag_1` and `volume_lag_1`. Drop the first row because no previous session is available. Keep the original trading dates without inserting weekends or holidays. Always save `microsoft_stock_prepared.csv`, replacing any existing output with the regenerated data in the same column order. Volume remains an integer count.


In [ ]:
ms_name = "microsoft_stock_prepared.csv"
microsoft = read_raw("Microsoft_Stock.csv", ["Date", "Open", "Close", "Volume"])
microsoft = dated(microsoft, "Date", "%m/%d/%Y %H:%M:%S")
microsoft = numeric(microsoft, ["Open", "Close", "Volume"])
if microsoft.index.has_duplicates or microsoft[["Open", "Close", "Volume"]].isna().any().any():
    raise ValueError("Microsoft: duplicate dates or missing numeric values")
if (microsoft["Volume"] < 0).any() or (microsoft["Volume"] % 1 != 0).any():
    raise ValueError("Microsoft: Volume must be a nonnegative integer count")
microsoft = pd.DataFrame({"target_open": microsoft["Open"],
                          "close_lag_1": microsoft["Close"].shift(1),
                          "volume_lag_1": microsoft["Volume"].shift(1)}).iloc[1:]
microsoft["volume_lag_1"] = microsoft["volume_lag_1"].astype("int64")
microsoft = save_series(ms_name, microsoft, "target_open", "observed trading sessions")
microsoft.head()


## Verify the saved files

Each saved series has unique, ascending dates and a finite numeric target. The summary lists the target and time unit needed by the main pipeline. Reload with `pd.read_csv(path, parse_dates=["Date"], index_col="Date")`; use `.asfreq("D")` for daily series and `.asfreq("YS")` for Brazil. Keep stock observations on their actual dates. Register the frame using the existing `add_base(nome=..., df=..., alvo=..., h=...)` function, choosing the forecast horizon for that series. No models are trained by this notebook.


In [ ]:
for item in summary:
    saved = pd.read_csv(OUT / item["file"], parse_dates=["Date"], index_col="Date")
    expected = prepared[item["file"]]
    assert len(saved) == len(expected)
    assert saved.index.is_monotonic_increasing and saved.index.is_unique
    assert saved.index.equals(expected.index)
    assert np.isfinite(saved[item["target"]]).all()
    assert np.allclose(saved[item["target"]], expected[item["target"]])

handled = {"Microsoft_Stock.csv", "DailyDelhiClimateTrain.csv", "DailyDelhiClimateTest.csv",
           stock_file, "brazil.csv", "Sales.csv"}
unhandled = {p.name for p in RAW.glob("*.csv")} - handled
if unhandled:
    raise ValueError(f"New raw CSVs need an explicit processing rule: {sorted(unhandled)}")
pd.DataFrame(summary)